# Berkshire Hathaway Sentiment Analysis

This notebook loads precomputed sentiment features, merges in SPY yearly returns, builds sentiment regimes, and evaluates whether sentiment has signal for the following year.


In [ ]:
import sys
from pathlib import Path
import importlib.util

import matplotlib.pyplot as plt
import pandas as pd

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))

from src.config.paths import FEATURES_DIR

features_path = FEATURES_DIR / "sentiment_features.csv"
print(f"Loading sentiment data from: {features_path}")

if not features_path.exists():
    raise FileNotFoundError(f"Sentiment file not found at {features_path}")

df = pd.read_csv(features_path)
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.dropna(subset=["year"]).copy()
df["year"] = df["year"].astype(int)
df = df.sort_values("year").reset_index(drop=True)

def load_spy_prices() -> pd.DataFrame:
    candidate_paths = [
        repo_root / "data" / "spy_prices.csv",
        repo_root / "data" / "prices" / "spy.csv",
        repo_root / "features" / "spy_prices.csv",
        repo_root / "output" / "spy_prices.csv",
    ]

    for candidate in candidate_paths:
        if candidate.exists():
            prices = pd.read_csv(candidate)
            print(f"Loaded SPY prices from local dataset: {candidate}")
            break
    else:
        if importlib.util.find_spec("yfinance") is None:
            raise ModuleNotFoundError(
                "No local SPY dataset was found and yfinance is not installed."
            )

        import yfinance as yf

        start_year = int(df["year"].min()) - 1
        end_year = int(df["year"].max()) + 1
        prices = yf.download(
            "SPY",
            start=f"{start_year}-01-01",
            end=f"{end_year}-12-31",
            progress=False,
            auto_adjust=False,
        ).reset_index()
        print("Downloaded SPY prices with yfinance")

    date_column = "Date" if "Date" in prices.columns else "date"
    close_column = "Close" if "Close" in prices.columns else "close"
    prices = prices.rename(columns={date_column: "date", close_column: "close"})
    prices["date"] = pd.to_datetime(prices["date"])
    prices["year"] = prices["date"].dt.year
    return prices[["date", "year", "close"]].dropna(subset=["close"]).copy()

prices = load_spy_prices()
returns = prices.groupby("year")["close"].last().pct_change().rename("returns")
df = df.merge(returns, on="year", how="left")
df.head()


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["year"], df["sentiment_score"], marker="o")
plt.title("Berkshire Hathaway Sentiment Over Time")
plt.xlabel("Year")
plt.ylabel("Sentiment Score")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
df["sentiment_ma"] = df["sentiment_score"].rolling(3).mean()
df["sentiment_regime"] = pd.cut(
    df["sentiment_score"],
    bins=[-1, -0.1, 0.1, 1],
    labels=["bearish", "neutral", "bullish"],
)

print(df["sentiment_regime"].value_counts())
df[["year", "sentiment_score", "sentiment_ma", "returns", "sentiment_regime"]].head()


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(df["year"], df["sentiment_score"], label="Raw", marker="o")
plt.plot(df["year"], df["sentiment_ma"], label="3yr MA", marker="o")
plt.legend()
plt.title("Sentiment Score and 3-Year Moving Average")
plt.xlabel("Year")
plt.ylabel("Sentiment Score")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["sentiment_score"], df["returns"], alpha=0.8)
plt.title("Sentiment Score vs. SPY Yearly Return")
plt.xlabel("Sentiment Score")
plt.ylabel("SPY Yearly Return")
plt.grid(alpha=0.3)
plt.show()

df["next_year_return"] = df["returns"].shift(-1)
print(df.groupby("sentiment_regime")["next_year_return"].mean())

print(df.nsmallest(5, "sentiment_score"))
